<a href="https://colab.research.google.com/github/vinkoff/Learner/blob/master/module5_Part2_governance_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚖️ Module 7 — Week 2: Debiasing, Governance & Emerging Risks
### Fixing Bias · NIST AI RMF · EU AI Act · Misinformation Detection

---

## 🎯 Week 2 Learning Objectives
By the end of this notebook you will be able to:
1. **Apply** threshold adjustment debiasing and measure its effect
2. **Interpret** the fairness-accuracy trade-off
3. **Complete** a NIST AI RMF compliance checklist for a hiring AI system
4. **Classify** a system under EU AI Act risk tiers
5. **Detect** misinformation indicators in AI-generated text

## 📋 Week 1 Recap
Last week we built a biased hiring dataset, trained a classifier,
and measured demographic parity and equal opportunity gaps.
This week we fix the bias and evaluate the system against real governance frameworks.

> **Important:** Run all cells from the top — Week 2 rebuilds the dataset and model
> so everything is ready for the debiasing and governance steps.

## ⚙️ Step 1 — Install & Import

In [1]:
!pip install anthropic scikit-learn matplotlib seaborn pandas numpy --quiet

import os, math, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import anthropic
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

np.random.seed(42)
print("✅ All libraries loaded!")

✅ All libraries loaded!


## 🔑 Step 2 — API Setup
> **Colab users:** Add `ANTHROPIC_API_KEY` to the 🔑 Secrets panel and toggle notebook access ON.

In [2]:
from google.colab import userdata
api_key = userdata.get("ANTHROPIC_API_KEY").strip()
client  = anthropic.Anthropic(api_key=api_key)
MODEL   = "claude-haiku-4-5"
print("✅ Client ready — using " + MODEL)

✅ Client ready — using claude-haiku-4-5


## 🔄 Step 3 — Rebuild Dataset & Model from Week 1

We recreate everything from Week 1 using the same random seed so results are identical.

In [3]:
# Healthcare domain — patient readmission risk
n = 1000

prior_admissions = np.random.randint(0, 6, n)
chronic_conditions = np.random.randint(0, 5, n)
length_of_stay = np.random.randint(1, 15, n)
med_count = np.random.randint(1, 12, n)
lab_abnormalities = np.random.randint(0, 8, n)

race = np.random.choice(['White','Black','Hispanic','Asian'], n, p=[0.55,0.18,0.18,0.09])
insurance = np.random.choice(['Private','Medicare','Medicaid'], n, p=[0.45,0.30,0.25])
age_group = np.random.choice(['Under 50','50-69','70+'], n, p=[0.30,0.40,0.30])

# Clinical risk score calculation
clinical_risk = (prior_admissions/5*0.25 + chronic_conditions/4*0.25 +
                 length_of_stay/14*0.20 + med_count/11*0.15 + lab_abnormalities/7*0.15)

# Bias: Medicaid, Black, and older patients artificially scored lower
bias = np.zeros(n)
bias += np.where(insurance == 'Medicaid', -0.15, 0)
bias += np.where(race == 'Black', -0.10, 0)
bias += np.where(age_group == '70+', -0.12, 0)

risk_prob = np.clip(clinical_risk + bias, 0.05, 0.95)
readmitted = np.random.binomial(1, risk_prob, n)

df = pd.DataFrame({
    "prior_admissions": prior_admissions, "chronic_conditions": chronic_conditions,
    "length_of_stay": length_of_stay, "med_count": med_count,
    "lab_abnormalities": lab_abnormalities, "race": race,
    "insurance": insurance, "age_group": age_group, "readmitted": readmitted
})

feature_cols = ["prior_admissions", "chronic_conditions", "length_of_stay", "med_count", "lab_abnormalities"]
X = df[feature_cols].values
y = df["readmitted"].values
indices = np.arange(len(df))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, indices, test_size=0.30, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=500)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
df_test = df.iloc[idx_test].copy()
df_test["predicted"] = y_pred

def demographic_parity(df_eval, protected_col, pred_col="predicted"):
    rates = df_eval.groupby(protected_col)[pred_col].mean()
    disparities = (rates.max() - rates).round(4)
    return rates.round(4), disparities

def equal_opportunity(df_eval, protected_col, label_col="readmitted", pred_col="predicted"):
    results = {}
    for group, subset in df_eval.groupby(protected_col):
        qualified = subset[subset[label_col] == 1]
        results[group] = round((qualified[pred_col] == 1).mean(), 4) if len(qualified) > 0 else None
    rates  = pd.Series(results)
    dispar = (rates.max() - rates).round(4)
    return rates, dispar

print("✅ Healthcare Dataset, model, and fairness functions ready!")
print("   Accuracy: " + f"{(y_pred == y_test).mean():.1%}")

✅ Healthcare Dataset, model, and fairness functions ready!
   Accuracy: 67.3%


## 🔧 Step 4 — Debiasing: Threshold Adjustment

**The Problem:** Our model uses a single threshold (0.5) for everyone.

**The Fix:** Use different thresholds per group to equalise the true positive rate.
This is post-processing — no retraining needed.

> **Trade-off Alert:** Equalising one metric often slightly reduces overall accuracy.
> This is the **fairness-accuracy trade-off** — one of the central tensions in AI ethics.

In [4]:
proba_test       = model.predict_proba(X_test)[:, 1]
df_test          = df_test.copy()
df_test["proba"] = proba_test

def find_threshold(group_df, target_tpr, label_col="readmitted"):
    qualified = group_df[group_df[label_col] == 1]
    if len(qualified) == 0:
        return 0.5
    best_t, best_diff = 0.5, 1.0
    for t in np.arange(0.1, 0.9, 0.01):
        tpr  = (qualified["proba"] >= t).mean()
        diff = abs(tpr - target_tpr)
        if diff < best_diff:
            best_diff, best_t = diff, t
    return round(best_t, 2)

race_tprs = {}
for race_val, subset in df_test.groupby("race"):
    q = subset[subset["readmitted"] == 1]
    race_tprs[race_val] = (q["proba"] >= 0.5).mean() if len(q) > 0 else 0

target_tpr = min(race_tprs.values())
print("Target TPR for equalisation: " + f"{target_tpr:.3f}")
print()

thresholds = {}
for race_val, subset in df_test.groupby("race"):
    t = find_threshold(subset, target_tpr)
    thresholds[race_val] = t
    print("  " + race_val.ljust(12) + " threshold: " + str(t) + "  (was 0.50)")

df_test["predicted_fair"] = df_test.apply(
    lambda row: 1 if row["proba"] >= thresholds[row["race"]] else 0, axis=1
)
print()
print("Debiased predictions generated!")

Target TPR for equalisation: 0.125

  Asian        threshold: 0.5  (was 0.50)
  Black        threshold: 0.54  (was 0.50)
  Hispanic     threshold: 0.69  (was 0.50)
  White        threshold: 0.66  (was 0.50)

Debiased predictions generated!


## 📊 Step 5 — Before vs After: Did Debiasing Help?

In [5]:
print("=" * 65)
print("FAIRNESS COMPARISON: BEFORE vs AFTER DEBIASING (Race)")
print("=" * 65)

for metric_name, fn, before_col, after_col in [
    ("Demographic Parity", demographic_parity, "predicted", "predicted_fair"),
    ("Equal Opportunity",  equal_opportunity,  "predicted", "predicted_fair"),
]:
    rates_b, disp_b = fn(df_test, "race", pred_col=before_col)
    rates_a, disp_a = fn(df_test, "race", pred_col=after_col)
    print()
    print(metric_name)
    print("  " + "Group".ljust(12) + "Before".rjust(10) + "After".rjust(10) + "Change".rjust(10))
    print("  " + "-"*45)
    for group in sorted(rates_b.index):
        change = rates_a[group] - rates_b[group]
        arrow  = "up" if change > 0 else ("dn" if change < 0 else "==")
        print("  " + group.ljust(12) + f"{rates_b[group]:>10.3f}{rates_a[group]:>10.3f}  {arrow} {abs(change):.3f}")
    improved = disp_a.max() < disp_b.max()
    print("  Max disparity: " + f"{disp_b.max():.3f} -> {disp_a.max():.3f}  " + ("IMPROVED" if improved else "NO CHANGE"))

# Replaced "hired" with "readmitted"
acc_before = (df_test["predicted"]      == df_test["readmitted"]).mean()
acc_after  = (df_test["predicted_fair"] == df_test["readmitted"]).mean()

print()
print("Accuracy:  " + f"{acc_before:.1%} -> {acc_after:.1%}  (change: {acc_after - acc_before:+.1%})")
print()
print("This is the fairness-accuracy trade-off.")
print("Improving fairness often slightly reduces overall accuracy.")
print("This is a design choice that must be made transparently.")

FAIRNESS COMPARISON: BEFORE vs AFTER DEBIASING (Race)

Demographic Parity
  Group           Before     After    Change
  ---------------------------------------------
  Asian            0.100     0.100  == 0.000
  Black            0.190     0.121  dn 0.069
  Hispanic         0.241     0.056  dn 0.185
  White            0.256     0.059  dn 0.197
  Max disparity: 0.156 -> 0.065  IMPROVED

Equal Opportunity
  Group           Before     After    Change
  ---------------------------------------------
  Asian            0.125     0.125  == 0.000
  Black            0.250     0.150  dn 0.100
  Hispanic         0.476     0.143  dn 0.333
  White            0.419     0.122  dn 0.297
  Max disparity: 0.351 -> 0.028  IMPROVED

Accuracy:  67.3% -> 62.3%  (change: -5.0%)

This is the fairness-accuracy trade-off.
Improving fairness often slightly reduces overall accuracy.
This is a design choice that must be made transparently.


## 🏛️ Step 6 — NIST AI Risk Management Framework

The **NIST AI RMF** (2023) has four core functions: **GOVERN → MAP → MEASURE → MANAGE**

### 📋 Checklist Instructions
Change each rating to one of:
- `"Met"` — we have this in place
- `"Partial"` — started but incomplete
- `"Not Met"` — not addressed
- `"Unknown"` — we don't know

Edit the ratings below then run the cell to see your compliance score.

In [6]:
checklist = {
 "GOVERN — Policies & Accountability": {
 "G1": ("Roles and responsibilities for AI risk are documented", "Met"),
 "G2": ("An AI ethics policy exists and covers healthcare systems", "Partial"),
 "G3": ("Affected patient stakeholders have been consulted", "Not Met"),
 "G4": ("A process exists to report and respond to clinical AI harms", "Partial"),
 "G5": ("AI decision-making is disclosed to patients", "Not Met"),
 },
 "MAP — Context & Risk Identification": {
 "M1": ("System purpose, scope, and limitations are documented", "Met"),
 "M2": ("Protected attributes and proxy variables are identified", "Partial"),
 "M3": ("Legal requirements (ACA 1557, HIPAA) have been reviewed", "Met"),
 "M4": ("Potential harms to patients have been catalogued", "Partial"),
 "M5": ("Training data bias has been assessed", "Not Met"),
 },
 "MEASURE — Testing & Evaluation": {
 "ME1": ("Demographic parity measured across all protected groups", "Partial"),
 "ME2": ("Equal opportunity measured across all protected groups", "Not Met"),
 "ME3": ("Model tested on data from multiple time periods", "Partial"),
 "ME4": ("Adverse impact ratio meets the EEOC 4/5ths rule", "Not Met"),
 "ME5": ("Regular re-audits are scheduled at least annually", "Partial"),
 },
 "MANAGE — Response & Monitoring": {
 "MG1": ("A process exists to appeal AI clinical decisions", "Not Met"),
 "MG2": ("Human review required before final discharge decisions", "Partial"),
 "MG3": ("A debiasing technique has been implemented and validated", "Not Met"),
 "MG4": ("Model is monitored in production for performance drift", "Partial"),
 "MG5": ("A decommission plan exists if the system causes harm", "Met"),
 }
}

icons     = {"Met": "[MET]", "Partial": "[PART]", "Not Met": "[FAIL]", "Unknown": "[?]"}
score_map = {"Met": 2, "Partial": 1, "Not Met": 0, "Unknown": 0}
total, max_score = 0, 0

print("=" * 70)
print("NIST AI RMF AUDIT — Healthcare AI System")
print("=" * 70)

for function, items in checklist.items():
    print()
    print(function)
    print("-" * 70)
    for code, (desc, rating) in items.items():
        icon       = icons.get(rating, "[?]")
        total     += score_map.get(rating, 0)
        max_score += 2
        print("  " + icon.ljust(7) + " [" + code + "] " + desc)
        print("          Rating: " + rating)

pct = round(total / max_score * 100, 1) if max_score > 0 else 0
print()
print("=" * 70)
print("SCORE: " + str(total) + "/" + str(max_score) + " (" + str(pct) + "%)")
if pct >= 75:
    print("Status: GOOD — Most controls in place")
elif pct >= 50:
    print("Status: MODERATE — Significant gaps remain")
else:
    print("Status: HIGH RISK — Must address gaps before deployment")
print("=" * 70)

NIST AI RMF AUDIT — Healthcare AI System

GOVERN — Policies & Accountability
----------------------------------------------------------------------
  [MET]   [G1] Roles and responsibilities for AI risk are documented
          Rating: Met
  [PART]  [G2] An AI ethics policy exists and covers healthcare systems
          Rating: Partial
  [FAIL]  [G3] Affected patient stakeholders have been consulted
          Rating: Not Met
  [PART]  [G4] A process exists to report and respond to clinical AI harms
          Rating: Partial
  [FAIL]  [G5] AI decision-making is disclosed to patients
          Rating: Not Met

MAP — Context & Risk Identification
----------------------------------------------------------------------
  [MET]   [M1] System purpose, scope, and limitations are documented
          Rating: Met
  [PART]  [M2] Protected attributes and proxy variables are identified
          Rating: Partial
  [MET]   [M3] Legal requirements (ACA 1557, HIPAA) have been reviewed
          Rating: M

## 🇪🇺 Step 7 — EU AI Act Risk Classification

The **EU AI Act** (2024) classifies AI into four risk tiers:

| Tier | Examples | Obligations |
|------|----------|-------------|
| 🔴 Unacceptable | Social scoring, subliminal manipulation | Prohibited |
| 🟠 High Risk | Hiring, credit, education, law enforcement | Conformity assessment, transparency, human oversight |
| 🟡 Limited Risk | Chatbots, deepfakes | Must disclose AI involvement |
| 🟢 Minimal Risk | Spam filters, games | No mandatory requirements |

**Hiring AI = High Risk (Annex III, Category 4)**

Change each rating to: `"Compliant"` | `"Partial"` | `"Non-Compliant"` | `"N/A"`

In [7]:
eu_checklist = {
 "Risk Management System (Art. 9)": {
 "EU1": ("A risk management system is established and maintained", "Partial"),
 "EU2": ("Risks from training data bias are identified and mitigated", "Non-Compliant"),
 "EU3": ("Residual risks are evaluated before deployment", "Non-Compliant"),
 },
 "Data Governance (Art. 10)": {
 "EU4": ("Training data is examined for biases", "Partial"),
 "EU5": ("Data is relevant, representative, and reasonably error-free", "Partial"),
 "EU6": ("Special category data race and health status is handled lawfully", "Non-Compliant"),
 },
 "Transparency and Documentation (Art. 11-13)": {
 "EU7": ("Technical documentation is prepared before market placement", "Partial"),
 "EU8": ("The system provides explainability to affected patients", "Non-Compliant"),
 "EU9": ("Patients are informed an AI system is being used", "Non-Compliant"),
 },
 "Human Oversight (Art. 14)": {
 "EU10": ("Humans can fully understand and monitor the system", "Partial"),
 "EU11": ("Humans can override or disable the system", "Partial"),
 "EU12": ("No automation bias — doctors make final clinical decisions", "Non-Compliant"),
 },
 "Accuracy and Robustness (Art. 15)": {
 "EU13": ("Accuracy metrics are documented and meet declared levels", "Partial"),
 "EU14": ("Fairness metrics are logged and reviewed regularly", "Partial"),
 "EU15": ("System is tested for adversarial inputs and distribution shift", "Non-Compliant"),
 }
}

eu_icons  = {"Compliant": "[YES]", "Partial": "[PART]", "Non-Compliant": "[NO]", "N/A": "[N/A]"}
eu_scores = {"Compliant": 2, "Partial": 1, "Non-Compliant": 0, "N/A": 1}
total, maximum = 0, 0

print("=" * 68)
print("EU AI ACT — HIGH RISK COMPLIANCE AUDIT (Health System)")
print("=" * 68)

for article, items in eu_checklist.items():
    print()
    print(article)
    print("-" * 68)
    for code, (desc, rating) in items.items():
        icon     = eu_icons.get(rating, "[?]")
        total   += eu_scores.get(rating, 0)
        maximum += 2
        print("  " + icon.ljust(7) + " [" + code + "] " + desc)
        print("          Status: " + rating)

pct = round(total / maximum * 100, 1) if maximum > 0 else 0
print()
print("=" * 68)
print("COMPLIANCE SCORE: " + str(total) + "/" + str(maximum) + " (" + str(pct) + "%)")
if pct >= 80:
    print("-> LIKELY COMPLIANT — Ready for conformity assessment")
elif pct >= 50:
    print("-> PARTIALLY COMPLIANT — Address gaps before deployment")
else:
    print("-> NON-COMPLIANT — Cannot be legally deployed in the EU")
print()
print("Non-compliance fines: up to 30M EUR or 6% of global annual turnover")
print("=" * 68)

EU AI ACT — HIGH RISK COMPLIANCE AUDIT (Health System)

Risk Management System (Art. 9)
--------------------------------------------------------------------
  [PART]  [EU1] A risk management system is established and maintained
          Status: Partial
  [NO]    [EU2] Risks from training data bias are identified and mitigated
          Status: Non-Compliant
  [NO]    [EU3] Residual risks are evaluated before deployment
          Status: Non-Compliant

Data Governance (Art. 10)
--------------------------------------------------------------------
  [PART]  [EU4] Training data is examined for biases
          Status: Partial
  [PART]  [EU5] Data is relevant, representative, and reasonably error-free
          Status: Partial
  [NO]    [EU6] Special category data race and health status is handled lawfully
          Status: Non-Compliant

Transparency and Documentation (Art. 11-13)
--------------------------------------------------------------------
  [PART]  [EU7] Technical documentation 

## ⚠️ Step 8 — Dual-Use Risks: Misinformation Detection

Beyond hiring bias, GenAI poses broader risks:

| Risk | Description |
|------|-------------|
| **Misinformation** | AI generates false but plausible content |
| **Deepfakes** | AI synthesises realistic but fake media |
| **Dual-Use** | Technology built for good, weaponised for harm |
| **Model Collapse** | AI trained on AI-generated data degrades over time |

We use Claude to demonstrate **detection** of misleading AI-generated content.

In [8]:
suspicious = (
    "BREAKING: Scientists at Harvard confirmed that daily coffee consumption "
    "completely prevents Alzheimers disease with 100% efficacy. The 30-year study "
    "of 50,000 participants found zero cases among coffee drinkers. The FDA is "
    "expected to approve coffee as an official treatment next month. "
    "Big Pharma is trying to suppress this finding."
)

detect_prompt = (
    "You are a misinformation detection specialist. "
    "Analyse the text below and identify:\n"
    "1. CREDIBILITY RED FLAGS - claims that are exaggerated or implausible\n"
    "2. EMOTIONAL MANIPULATION - language designed to trigger fear or outrage\n"
    "3. MISSING CONTEXT - what a legitimate source would include\n"
    "4. MALICIOUS CONTENT - language contains harmful content"
    "5. VERDICT - rate credibility: High / Medium / Low / Very Low\n"
    "6. RECOMMENDED ACTION - what should a reader do before sharing?\n\n"
    "Text:\n" + suspicious
)

print("Sending suspicious text to Claude for misinformation analysis...")
response = client.messages.create(
    model=MODEL, max_tokens=800,
    messages=[{"role": "user", "content": detect_prompt}]
)
print()
print("=" * 60)
print("MISINFORMATION DETECTION REPORT")
print("=" * 60)
print(response.content[0].text)

print("Sending suspicious text to Claude for misinformation analysis...")
response = client.messages.create(
    model=MODEL, max_tokens=800,
    messages=[{"role": "user", "content": detect_prompt}]
)
print()
print("=" * 60)
print("MISINFORMATION DETECTION REPORT")
print("=" * 60)
print(response.content[0].text)

Sending suspicious text to Claude for misinformation analysis...

MISINFORMATION DETECTION REPORT
# Misinformation Analysis

## 1. CREDIBILITY RED FLAGS ⚠️

- **"Completely prevents...with 100% efficacy"** - No medical intervention is 100% effective. This absolute claim is implausible.
- **"Zero cases among coffee drinkers"** - Statistically improbable in a large study; suggests data manipulation or fabrication.
- **"FDA is expected to approve coffee as an official treatment next month"** - Coffee is already widely consumed; the FDA doesn't "approve" common beverages as treatments. This is factually incorrect.
- **Vague sourcing** - No specific researchers, institution details, or study publication cited.
- **No peer review mention** - A major finding would reference journal publication, which is absent.

## 2. EMOTIONAL MANIPULATION 😠

- **"BREAKING"** - Creates false urgency
- **"Big Pharma is trying to suppress this finding"** - Classic conspiracy framing designed to:
  - Trigger di

## 💡 Step 9 — Key Takeaways & Discussion

### 🤔 Discussion Questions

1. **Fairness-accuracy trade-off:** Our debiasing slightly reduced accuracy.
   Who bears the cost of that accuracy drop? Is it worth it?

2. **Proxy variables:** Removing protected attributes from features is not enough.
   What would it truly take to remove bias from a model trained on historical data?

3. **NIST vs EU AI Act:** NIST is voluntary; the EU AI Act is law.
   Should governance frameworks be mandatory? Who should enforce them?

4. **Dual-use:** Claude can both generate AND detect misleading content.
   Does that make it more dangerous or more useful? Who is responsible for misuse?

5. **The auditor problem:** We used an AI (Claude) to audit another AI system.
   What are the risks of using AI to govern AI?

---

## 🎓 Module 7 Assignment
Pick **one domain** from the list below and repeat the full two-week pipeline:
- 🏥 Healthcare — clinical risk scoring
- ⚖️ Criminal Justice — recidivism prediction
- 💰 Finance — loan approval
- 🎓 Education — admissions scoring
- 📱 Content Moderation — toxicity classification

**Deliverables:**
1. Working notebook with your domain data, fairness metrics, debiasing, and both governance checklists
2. Results JSON file
3. Written reflection (400-600 words) answering all five discussion questions above
4. AI collaboration log — what you asked AI to help with and what you changed

In [9]:
with open("week2_results.json", "w") as f:
    json.dump({
        "acc_before_debiasing": float((df_test["predicted"]      == df_test["readmitted"]).mean()),
        "acc_after_debiasing":  float((df_test["predicted_fair"] == df_test["readmitted"]).mean()),
        "thresholds_by_race":   thresholds,
    }, f, indent=2)

print("Results saved to week2_results.json")
print()
print("Module 7 complete!")
print("You have: detected bias, measured fairness, debiased a model,")
print("applied NIST AI RMF and EU AI Act frameworks, and detected misinformation.")

Results saved to week2_results.json

Module 7 complete!
You have: detected bias, measured fairness, debiased a model,
applied NIST AI RMF and EU AI Act frameworks, and detected misinformation.
